In [1]:
import polars as pl
from pathlib import Path
from datetime import datetime, timezone

DATA_DIR = Path('../data/processed')
WIDE = DATA_DIR / 'wide'

trainA = pl.read_parquet(WIDE / 'trainA.parquet')
valA   = pl.read_parquet(WIDE / 'valA.parquet')
testA  = pl.read_parquet(WIDE / 'testA.parquet')
trainB = pl.read_parquet(WIDE / 'trainB.parquet')
valB   = pl.read_parquet(WIDE / 'valB.parquet')
testB  = pl.read_parquet(WIDE / 'testB.parquet')

print(f'trainA: {trainA.shape}')
print(f'trainB: {trainB.shape}')

trainA: (20437223, 26)
trainB: (19281955, 25)


In [2]:
# Anti-leakage cutoff: end of train period (UTC 2017-05-12 00:00).
# Aggregate features only use behavior data BEFORE this point,
# so val/test impressions cannot peek at concurrent user behaviors.
trainEnd = int(datetime(2017, 5, 12, 0, 0, 0, tzinfo=timezone.utc).timestamp())
validStart = int(datetime(2017, 4, 1).timestamp())

# One-pass group-by: compute 6 raw aggregates per user from behavior_log.
behaviorAgg = (
    pl.scan_parquet(DATA_DIR / 'behavior_log.parquet')
    .filter(
        (pl.col('time_stamp') >= validStart) &
        (pl.col('time_stamp') < trainEnd)
    )
    .group_by('user')
    .agg(
        user_total_events=pl.len(),
        user_total_pv=(pl.col('btag') == 'pv').sum(),
        user_total_buy=(pl.col('btag') == 'buy').sum(),
        user_total_cart=(pl.col('btag') == 'cart').sum(),
        user_total_fav=(pl.col('btag') == 'fav').sum(),
        user_n_unique_cates=pl.col('cate').n_unique(),
    )
    .collect()
)

# Global buy rate over the train-period behavior log; used as the prior
# for Bayesian smoothing of per-user buy rate.
totalBuy = behaviorAgg['user_total_buy'].sum()
totalEvents = behaviorAgg['user_total_events'].sum()
globalBuyRate = totalBuy / totalEvents
print(f'Global buy rate: {globalBuyRate:.4f}')

# Bayesian smoothing: pull low-activity users toward the global rate.
# alpha controls smoothing strength: smaller alpha = more user-specific,
# larger alpha = more global. alpha=10 is a typical CTR-modeling default.
ALPHA = 10
behaviorAgg = behaviorAgg.with_columns(
    user_buy_rate=(
        (pl.col('user_total_buy') + ALPHA * globalBuyRate) /
        (pl.col('user_total_events') + ALPHA)
    )
)

print(f'\nbehaviorAgg shape: {behaviorAgg.shape}')
print(behaviorAgg.head())

Global buy rate: 0.0127

behaviorAgg shape: (1132067, 8)
shape: (5, 8)
┌─────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────────┬───────────┐
│ user    ┆ user_total ┆ user_total ┆ user_total ┆ user_total ┆ user_total ┆ user_n_un ┆ user_buy_ │
│ ---     ┆ _events    ┆ _pv        ┆ _buy       ┆ _cart      ┆ _fav       ┆ ique_cate ┆ rate      │
│ i64     ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ s         ┆ ---       │
│         ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆ u32        ┆ ---       ┆ f64       │
│         ┆            ┆            ┆            ┆            ┆            ┆ u32       ┆           │
╞═════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╡
│ 101612  ┆ 289        ┆ 271        ┆ 11         ┆ 7          ┆ 0          ┆ 44        ┆ 0.037213  │
│ 1047987 ┆ 1336       ┆ 1295       ┆ 6          ┆ 18         ┆ 17         ┆ 94        ┆ 0.004552  │
│ 31351   ┆ 420     

In [3]:
def joinBehaviorAgg(df, behaviorAgg, globalBuyRate):
    """Left-join per-user behavior aggregates onto an impression dataframe.

    Users absent from behavior_log (cold-start) get 0 for count features
    and the global prior for the rate feature.
    """
    return (
        df.join(behaviorAgg, on='user', how='left')
        .with_columns([
            pl.col('user_total_events').fill_null(0),
            pl.col('user_total_pv').fill_null(0),
            pl.col('user_total_buy').fill_null(0),
            pl.col('user_total_cart').fill_null(0),
            pl.col('user_total_fav').fill_null(0),
            pl.col('user_n_unique_cates').fill_null(0),
            pl.col('user_buy_rate').fill_null(globalBuyRate),
        ])
    )

trainA = joinBehaviorAgg(trainA, behaviorAgg, globalBuyRate)
valA   = joinBehaviorAgg(valA, behaviorAgg, globalBuyRate)
testA  = joinBehaviorAgg(testA, behaviorAgg, globalBuyRate)
trainB = joinBehaviorAgg(trainB, behaviorAgg, globalBuyRate)
valB   = joinBehaviorAgg(valB, behaviorAgg, globalBuyRate)
testB  = joinBehaviorAgg(testB, behaviorAgg, globalBuyRate)

print('Joined shapes:')
for name, df in [('trainA', trainA), ('valA', valA), ('testA', testA),
                  ('trainB', trainB), ('valB', valB), ('testB', testB)]:
    print(f'  {name}: {df.shape}')

Joined shapes:
  trainA: (20437223, 33)
  valA: (3271268, 33)
  testA: (2848377, 33)
  trainB: (19281955, 32)
  valB: (3078536, 32)
  testB: (2667885, 32)


In [4]:
# Sanity check: look at the new column distributions on trainA.
print('user_total_pv distribution (trainA):')
print(trainA.select('user_total_pv').describe())

# Overwrite the previously saved files in data/processed/wide/.
datasets = {'trainA': trainA, 'valA': valA, 'testA': testA,
            'trainB': trainB, 'valB': valB, 'testB': testB}

for name, df in datasets.items():
    path = WIDE / f'{name}.parquet'
    df.write_parquet(path, compression='snappy')
    sizeMb = path.stat().st_size / (1024**2)
    print(f'{name}: {df.shape} -> {sizeMb:.1f} MB')

user_total_pv distribution (trainA):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ user_total_pv │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 2.0437223e7   │
│ null_count ┆ 0.0           │
│ mean       ┆ 556.033754    │
│ std        ┆ 706.362028    │
│ min        ┆ 0.0           │
│ 25%        ┆ 138.0         │
│ 50%        ┆ 331.0         │
│ 75%        ┆ 703.0         │
│ max        ┆ 24368.0       │
└────────────┴───────────────┘
trainA: (20437223, 33) -> 732.0 MB
valA: (3271268, 33) -> 126.1 MB
testA: (2848377, 33) -> 107.8 MB
trainB: (19281955, 32) -> 688.2 MB
valB: (3078536, 32) -> 118.4 MB
testB: (2667885, 32) -> 100.9 MB


In [5]:
# Bayesian smoothing parameter (same alpha as user_buy_rate for consistency)
ALPHA_CTR = 10

# Global CTR from train; used as prior for unseen (user, cate) pairs
globalCtrA = trainA['clk'].mean()
print(f'Version A global CTR: {globalCtrA:.4f}')

# Aggregate (user, cate) → impressions + clicks, then smooth
userCateCtrA = (
    trainA
    .group_by(['user', 'cate_id'])
    .agg(
        n_imp=pl.len(),
        n_clk=pl.col('clk').sum(),
    )
    .with_columns(
        user_cate_ctr=(
            (pl.col('n_clk') + ALPHA_CTR * globalCtrA) /
            (pl.col('n_imp') + ALPHA_CTR)
        )
    )
    .drop(['n_imp', 'n_clk'])
)

print(f'Unique (user, cate) pairs in trainA: {userCateCtrA.height:,}')
print(userCateCtrA.head())

Version A global CTR: 0.0519
Unique (user, cate) pairs in trainA: 5,487,344
shape: (5, 3)
┌────────┬─────────┬───────────────┐
│ user   ┆ cate_id ┆ user_cate_ctr │
│ ---    ┆ ---     ┆ ---           │
│ u32    ┆ u32     ┆ f64           │
╞════════╪═════════╪═══════════════╡
│ 608868 ┆ 6637    ┆ 0.138108      │
│ 928036 ┆ 3545    ┆ 0.039937      │
│ 243398 ┆ 5028    ┆ 0.047199      │
│ 149053 ┆ 2785    ┆ 0.047199      │
│ 564299 ┆ 1529    ┆ 0.043266      │
└────────┴─────────┴───────────────┘


In [6]:
globalCtrB = trainB['clk'].mean()
print(f'Version B global CTR: {globalCtrB:.4f}')

userCateCtrB = (
    trainB
    .group_by(['user', 'cate_id'])
    .agg(
        n_imp=pl.len(),
        n_clk=pl.col('clk').sum(),
    )
    .with_columns(
        user_cate_ctr=(
            (pl.col('n_clk') + ALPHA_CTR * globalCtrB) /
            (pl.col('n_imp') + ALPHA_CTR)
        )
    )
    .drop(['n_imp', 'n_clk'])
)

print(f'Unique (user, cate) pairs in trainB: {userCateCtrB.height:,}')

Version B global CTR: 0.0518
Unique (user, cate) pairs in trainB: 5,169,445


In [7]:
def joinUserCateCtr(df, userCateCtr, globalCtr):
    """Left-join smoothed user-category CTR onto a split.

    (user, cate) pairs absent from the train statistics fall back to
    the global training CTR.
    """
    return (
        df.join(userCateCtr, on=['user', 'cate_id'], how='left')
        .with_columns(pl.col('user_cate_ctr').fill_null(globalCtr))
    )

# Apply to each split using the corresponding version's stats
trainA = joinUserCateCtr(trainA, userCateCtrA, globalCtrA)
valA   = joinUserCateCtr(valA, userCateCtrA, globalCtrA)
testA  = joinUserCateCtr(testA, userCateCtrA, globalCtrA)
trainB = joinUserCateCtr(trainB, userCateCtrB, globalCtrB)
valB   = joinUserCateCtr(valB, userCateCtrB, globalCtrB)
testB  = joinUserCateCtr(testB, userCateCtrB, globalCtrB)

print('After join shapes:')
for name, df in [('trainA', trainA), ('valA', valA), ('testA', testA),
                  ('trainB', trainB), ('valB', valB), ('testB', testB)]:
    print(f'  {name}: {df.shape}')

After join shapes:
  trainA: (20437223, 34)
  valA: (3271268, 34)
  testA: (2848377, 34)
  trainB: (19281955, 33)
  valB: (3078536, 33)
  testB: (2667885, 33)


In [8]:
# Inspect the new column distribution
print('user_cate_ctr distribution (trainA):')
print(trainA.select('user_cate_ctr').describe())

# Overwrite previously saved files
datasets = {'trainA': trainA, 'valA': valA, 'testA': testA,
            'trainB': trainB, 'valB': valB, 'testB': testB}

for name, df in datasets.items():
    path = WIDE / f'{name}.parquet'
    df.write_parquet(path, compression='snappy')
    sizeMb = path.stat().st_size / (1024**2)
    print(f'{name}: {df.shape} -> {sizeMb:.1f} MB')

user_cate_ctr distribution (trainA):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ user_cate_ctr │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 2.0437223e7   │
│ null_count ┆ 0.0           │
│ mean       ┆ 0.050469      │
│ std        ┆ 0.036726      │
│ min        ┆ 0.00102       │
│ 25%        ┆ 0.028844      │
│ 50%        ┆ 0.0422        │
│ 75%        ┆ 0.052386      │
│ max        ┆ 0.596892      │
└────────────┴───────────────┘
trainA: (20437223, 34) -> 790.1 MB
valA: (3271268, 34) -> 134.5 MB
testA: (2848377, 34) -> 114.8 MB
trainB: (19281955, 33) -> 743.0 MB
valB: (3078536, 33) -> 126.4 MB
testB: (2667885, 33) -> 107.5 MB
